# Mission 2 최종 결과와 과거 실험 구분

최종 설정은 각 모델 1/3 균등 평균, 임계값 0.50이다. Git 3638dab의 기존 평가 출력은 Validation 111,919건에서 정확도 92.48%, Macro F1 0.9244를 기록한다.

Git bf38dd1의 가중 앙상블은 ECAPA 0.45 / ReDim 0.40 / ResNet 0.15로, 임계값 0.50에서 92.57%, 0.51에서 92.58%와 Macro F1 0.9254였다. 최종 성적과 섞어 쓰지 않는다. Git 2c6cb4a에는 ECAPA FT 3에폭 완료 성적 91.82%가 있고, d6cdee7에는 Wav2Vec2 5에폭 완료 성적 89.72%가 있다. 원래 실행 노트북은 Docs/review_2026-10-08/historical_notebooks에 보존했다.

아래 코드 셀 출력은 Git 3638dab의 과거 실행 기록이다. 이번 코드 수정 후 새로 실행한 성적이 아니다. 최초 classification_report의 이름 순서는 반대였으므로 숫자 라벨 기준 첫 행은 상황실 0, 둘째 행은 신고자 1로 읽는다. 다음 실행의 코드는 이 이름을 바로잡고 예측 배열과 샘플 목록을 저장한다.

현재 Mac 작업 폴더에는 원본 평가 음성·라벨이 없어 수정한 코드의 새 전수 추론은 실행하지 않았다. Git에 보존된 과거 평가 결과와 변경 이력은 검토했다.

## 원격 a8fef84의 전처리 비교 요약 반영

원격 커밋에 추가된 전처리 요약은 Git 3638dab의 model_train.ipynb 초기 실험 출력과 일치한다. 아래 표는 그 과거 결과의 요약이며 이번 재실행 결과가 아니다.

| 조건 | 윈도우 | 패딩 | FFT | 최고 정확도 | Macro F1 |
|---|---|---|---|---|---|
| Pre-1 | 1.5초 | zero | 2048 | 84.92% | 0.8480 |
| Pre-2 | 3.0초 | repeat | 2048 | 85.14% | 0.8503 |
| Pre-3 | 3.0초 | zero | 2048 | 85.93% | 0.8584 |
| Pre-4 | 3.0초 | zero | 1024 | 85.48% | 0.8545 |

Training 30,313건, Validation 9,138건, 각 조건 3에폭의 기록이다. 모든 조건의 Mel은 128이다. 후속 모델별 FFT 설정과 구분한다.

같은 원격 커밋의 VRAM/OOM 프로파일링 요약은 실제 메모리 계측·학습·오류를 발생시키는 코드와 대응하지 않는다. 해당 수치를 검증된 실측으로 사용하지 않는다. 원격 원본은 Git a8fef84에 남아 있다.


In [2]:
import os
import glob
import json
import random
import librosa
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet50, ResNet50_Weights
from torch.cuda.amp import GradScaler, autocast
from torch.optim.lr_scheduler import CosineAnnealingLR
import warnings
from tqdm import tqdm

warnings.filterwarnings('ignore')

# 모델 선택: 'redimnet', 'ecapa_tdnn', 또는 'resnet50'
MODEL_TYPE = 'redimnet' # Part 1 ReDimNet 파이프라인 설정

# 데이터 경로 (로컬 환경 맞춤)
TRAIN_DIR = '../data/train'
VAL_DIR = '../data/val'

EPOCHS = 10
BATCH_SIZE = 32

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[확인] 사용 디바이스: {device}")
print(f"[확인] 선택된 모델: {MODEL_TYPE}")

# 모델별 최적 설정 자동 할당
if MODEL_TYPE == 'resnet50':
  N_MELS, N_FFT, HOP_LENGTH = 128, 2048, 512
elif MODEL_TYPE in ['redimnet', 'ecapa_tdnn']:
  N_MELS, N_FFT, HOP_LENGTH = 80, 512, 160

print(f"[확인] 음향 특징 설정: n_mels={N_MELS}, n_fft={N_FFT}, hop_length={HOP_LENGTH}")


[확인] 사용 디바이스: cuda
[확인] 선택된 모델: redimnet
[확인] 음향 특징 설정: n_mels=80, n_fft=512, hop_length=160


In [3]:
class DCCAudioDataset(Dataset):
  def __init__(self, data_dir, is_train=True, max_files=None, n_mels=128, n_fft=2048, hop_length=512, window_sec=3.0, padding_mode="zero"):
    self.data_dir = data_dir
    self.is_train = is_train
    self.n_mels = n_mels
    self.n_fft = n_fft
    self.hop_length = hop_length
    self.window_sec = window_sec
    self.padding_mode = padding_mode
    self.sr = 16000
    self.target_samples = int(self.sr * self.window_sec)
    
    self.samples = self._parse_data(max_files)
    
  def _parse_data(self, max_files):
    samples = []
    label_dir = os.path.join(self.data_dir, 'label')
    audio_dir = os.path.join(self.data_dir, 'audio')
    
    # Fallback 구조
    if not os.path.exists(label_dir):
      label_dir = self.data_dir 
      audio_dir = self.data_dir 
      
    json_files = glob.glob(os.path.join(label_dir, '**', '*.json'), recursive=True)
    if max_files:
      json_files = json_files[:max_files]
      
    for jf in json_files:
      with open(jf, 'r', encoding='utf-8') as f:
        data = json.load(f)
      
      # 음성 파일 경로 매핑
      wav_name = os.path.basename(jf).replace('.json', '.wav')
      wav_path = os.path.join(audio_dir, wav_name)
      
      if not os.path.exists(wav_path):
        wav_search = glob.glob(os.path.join(audio_dir, '**', wav_name), recursive=True)
        if wav_search:
          wav_path = wav_search[0]
        else:
          continue 
          
      if 'utterances' in data:
        for utt in data['utterances']:
          start_at = float(utt.get('startAt', 0))
          end_at = float(utt.get('endAt', 0))
          speaker = int(utt.get('speaker', 0))
          
          samples.append({
            'wav_path': wav_path,
            'start_ms': start_at,
            'end_ms': end_at,
            'label': speaker
          })
    return samples
    
  def __len__(self):
    return len(self.samples)
    
  def __getitem__(self, idx):
    sample = self.samples[idx]
    wav_path = sample['wav_path']
    
    # 1. 오디오 로드 (특정 구간) - 안전한 librosa 로직
    try:
      start_sec = sample['start_ms'] / 1000.0
      end_sec = sample['end_ms'] / 1000.0
      clip_dur = max(0.01, end_sec - start_sec)
      
      y, sr = librosa.load(wav_path, sr=self.sr, offset=start_sec, duration=clip_dur)
    except Exception:
      y = np.zeros(self.target_samples, dtype=np.float32)
      
    # 2. 오디오 길이 맞춤 (Padding or Crop)
    cur_len = len(y)
    if cur_len < self.target_samples:
      pad_needed = self.target_samples - cur_len
      if self.padding_mode == "repeat" and cur_len > 0:
        repeats = int(np.ceil(self.target_samples / cur_len))
        y = np.tile(y, repeats)[:self.target_samples]
      else: # zero padding
        y = np.pad(y, (0, pad_needed), mode='constant')
    elif cur_len > self.target_samples:
      if self.is_train:
        max_start = cur_len - self.target_samples
        start_idx = random.randint(0, max_start)
      else:
        start_idx = (cur_len - self.target_samples) // 2
      y = y[start_idx : start_idx + self.target_samples]

    # 3. Safe STFT 방어: 길이가 n_fft보다 작으면 추가 패딩
    if len(y) < self.n_fft:
      y = np.pad(y, (0, self.n_fft - len(y)), mode='constant')

    # 4. Mel-Spectrogram 변환 (dB scale)
    mel = librosa.feature.melspectrogram(
      y=y, sr=self.sr, n_fft=self.n_fft,
      hop_length=self.hop_length, n_mels=self.n_mels
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)

    # 5. 정규화 (Min-Max) & 1채널 텐서 변환
    mel_norm = (mel_db + 80.0) / 80.0
    mel_norm = np.clip(mel_norm, 0.0, 1.0)
    
    tensor_x = torch.tensor(mel_norm, dtype=torch.float32).unsqueeze(0) # (1, n_mels, time)
    tensor_y = torch.tensor(sample['label'], dtype=torch.float32)
      
    return tensor_x, tensor_y


In [4]:
def get_audio_resnet50(dropout_rate=0.3):
  model = resnet50(weights=ResNet50_Weights.DEFAULT)
  
  # 3채널 -> 1채널 가중치 평균 변환
  old_conv = model.conv1
  new_conv = nn.Conv2d(1, old_conv.out_channels, kernel_size=old_conv.kernel_size,
             stride=old_conv.stride, padding=old_conv.padding, bias=False)
  new_conv.weight.data = old_conv.weight.data.mean(dim=1, keepdim=True)
  model.conv1 = new_conv
  
  # FC 레이어 수정 (1 클래스 - BCEWithLogitsLoss 용)
  in_features = model.fc.in_features
  model.fc = nn.Sequential(
    nn.Dropout(dropout_rate),
    nn.Linear(in_features, 1)
  )
  return model

from m2.models import ReDimNet2_B2
from m2.models import ECAPA_TDNN

def build_selected_model(model_type):
  if model_type == 'resnet50':
    return get_audio_resnet50()
  elif model_type == 'redimnet':
    # ReDimNet도 1개의 logit을 출력하도록 설정 (BCEWithLogitsLoss 용)
    return ReDimNet2_B2(num_classes=1)
  elif model_type == 'ecapa_tdnn':
    return ECAPA_TDNN(in_channels=80, channels=512, num_classes=1)
  else:
    raise ValueError("Invalid model_type. Choose 'resnet50', 'redimnet', or 'ecapa_tdnn'.")


In [5]:
# ==========================================
# Sanity Check (데이터 로더 & 모델 순전파/역전파 검증)
# ==========================================
print("--- [Sanity Check 시작] ---")

# 1. 소규모 데이터로 데이터로더 생성 (최대 10개 파일만 파싱)
sanity_dataset = DCCAudioDataset(TRAIN_DIR, is_train=True, max_files=10, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
if len(sanity_dataset) == 0:
  print(f"[경고] 경고: '{TRAIN_DIR}' 에서 샘플을 찾지 못했습니다. 경로에 데이터가 있는지 확인하세요.")
else:
  sanity_loader = DataLoader(sanity_dataset, batch_size=2, shuffle=True)
  
  # 2. 모델 및 옵티마이저 생성
  sanity_model = build_selected_model(MODEL_TYPE).to(device)
  criterion = nn.BCEWithLogitsLoss()
  optimizer = optim.Adam(sanity_model.parameters(), lr=1e-4)
  scaler = GradScaler()
  
  sanity_model.train()
  
  try:
    inputs, labels = next(iter(sanity_loader))
    inputs = inputs.to(device)
    labels = labels.to(device).unsqueeze(1)
    
    print(f"입력 텐서 모양: {inputs.shape}")
    print(f"라벨 텐서 모양: {labels.shape}")
    
    optimizer.zero_grad()
    with autocast():
      outputs = sanity_model(inputs)
      loss = criterion(outputs, labels)
      
    print(f"모델 출력 모양: {outputs.shape}")
    print(f"손실값 (Loss): {loss.item():.4f}")
    
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    
    if torch.isnan(loss):
      print("[실패] 오류: Loss가 NaN입니다! 입력 데이터나 정규화를 확인하세요.")
    else:
      print("[확인] Forward/Backward 패스 성공! 모델 학습이 정상적으로 가능합니다.")
      
  except Exception as e:
    print(f"[실패] Sanity Check 실패: {e}")
    
print("--- [Sanity Check 종료] ---")


--- [Sanity Check 시작] ---
입력 텐서 모양: torch.Size([2, 1, 80, 301])
라벨 텐서 모양: torch.Size([2, 1])
모델 출력 모양: torch.Size([2, 1])
손실값 (Loss): 0.6781
[확인] Forward/Backward 패스 성공! 모델 학습이 정상적으로 가능합니다.
--- [Sanity Check 종료] ---


In [ ]:
def train_model():
  print("="*60)
  print(f" [RTX 3060 전용] {MODEL_TYPE} Full Training Pipeline (90%+ 버전)")
  print(f" - Train Dir: {TRAIN_DIR}")
  print(f" - Val Dir: {VAL_DIR}")
  print(f" - Batch Size: {BATCH_SIZE} | Epochs: {EPOCHS}")
  print(f" - AMP 활성화 | BCEWithLogitsLoss + Random Crop")
  print("="*60)
  
  if not os.path.exists(TRAIN_DIR):
    print(f"[경고] 에러: 학습 데이터 폴더 '{TRAIN_DIR}'를 찾을 수 없습니다. 경로를 확인해주세요.")
    return

  train_dataset = DCCAudioDataset(TRAIN_DIR, is_train=True, max_files=None, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
  val_dataset = DCCAudioDataset(VAL_DIR, is_train=False, max_files=None, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
  
  train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
  val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
  
  print(f"[확인] 데이터 로드 완료! (Train: {len(train_dataset):,} 샘플 | Val: {len(val_dataset):,} 샘플)")

  model = build_selected_model(MODEL_TYPE).to(device)
  
  # ---------------------------------------------------------
  # [핵심] 이전 벤치마크/학습에서 저장된 가중치 불러오기
  # ---------------------------------------------------------
  checkpoint_path = f"test_results/checkpoints/best_{MODEL_TYPE}.pt"
  if os.path.exists(checkpoint_path):
    print(f" 저장된 가중치 발견! 불러옵니다: {checkpoint_path}")
    try:
      checkpoint = torch.load(checkpoint_path, map_location=device)
      if 'model_state_dict' in checkpoint:
        state_dict = checkpoint['model_state_dict']
      else:
        state_dict = checkpoint
      
      # 크기가 일치하는 가중치만 안전하게 필터링하여 덮어쓰기 (Shape mismatch 원천 차단!)
      model_dict = model.state_dict()
      pretrained_dict = {k: v for k, v in state_dict.items() if k in model_dict and v.shape == model_dict[k].shape}
      model_dict.update(pretrained_dict)
      model.load_state_dict(model_dict)
      
      print("[확인] 가중치 로드 성공! (분류기를 제외한 핵심 특징 추출기 완벽 복구 완료)")
    except Exception as e:
      print(f"[경고] 가중치 로드 실패: {e}")
  else:
    print(f" 저장된 가중치가 없습니다 ({checkpoint_path}). 바닥부터 새로 학습합니다.")
  # ---------------------------------------------------------

  criterion = nn.BCEWithLogitsLoss()
  optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
  scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
  
  scaler = GradScaler()
  best_acc = 0.0
  os.makedirs('checkpoints', exist_ok=True)
  
  for epoch in range(1, EPOCHS + 1):
    # --- Training ---
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]")
    for inputs, labels in pbar:
      inputs, labels = inputs.to(device), labels.to(device).unsqueeze(1)
      optimizer.zero_grad()
      
      with autocast():
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
      scaler.scale(loss).backward()
      scaler.step(optimizer)
      scaler.update()
      
      train_loss += loss.item() * inputs.size(0)
      
      # 예측값 계산 (Sigmoid >= 0.5)
      preds = (torch.sigmoid(outputs) >= 0.5).float()
      train_total += labels.size(0)
      train_correct += preds.eq(labels).sum().item()
      pbar.set_postfix({'loss': f"{loss.item():.4f}"})
      
    scheduler.step()
    train_acc = train_correct / train_total
    train_loss = train_loss / train_total
    
    # --- Validation ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
      for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [Val]"):
        inputs, labels = inputs.to(device), labels.to(device).unsqueeze(1)
        
        with autocast():
          outputs = model(inputs)
          loss = criterion(outputs, labels)
          
        val_loss += loss.item() * inputs.size(0)
        preds = (torch.sigmoid(outputs) >= 0.5).float()
        val_total += labels.size(0)
        val_correct += preds.eq(labels).sum().item()
        
    val_acc = val_correct / val_total
    val_loss = val_loss / val_total
    
    print(f" Epoch {epoch:02d} 결과 | Train Acc: {train_acc*100:.2f}% | Val Loss: {val_loss:.4f} Val Acc: {val_acc*100:.2f}%")
    
    if val_acc > best_acc:
      best_acc = val_acc
      save_path = f"checkpoints/best_{MODEL_TYPE}.pt"
      torch.save(model.state_dict(), save_path)
      print(f"[추천] [신기록 달성!] 가중치 저장 완료: {save_path} (Acc: {best_acc*100:.2f}%)")


In [6]:
# 전체 데이터 풀 학습 시작
# 주의: 시간이 오래 걸릴 수 있으므로 Sanity Check가 통과된 이후에만 실행하세요.
train_model()


 [RTX 3060 전용] redimnet Full Training Pipeline (90%+ 버전)
 - Train Dir: ../data/train
 - Val Dir: ../data/val
 - Batch Size: 32 | Epochs: 10
 - AMP 활성화 | BCEWithLogitsLoss + Random Crop
[확인] 데이터 로드 완료! (Train: 873,137 샘플 | Val: 111,919 샘플)
 저장된 가중치 발견! 불러옵니다: test_results/checkpoints/best_redimnet.pt
[확인] 가중치 로드 성공! (분류기를 제외한 핵심 특징 추출기 완벽 복구 완료)


Epoch 1/10 [Val]: 100%|██████████| 3498/3498 [11:36<00:00, 5.02it/s]


 Epoch 01 결과 | Train Acc: 87.83% | Val Loss: 0.2123 Val Acc: 90.01%
[추천] [신기록 달성!] 가중치 저장 완료: checkpoints/best_redimnet.pt (Acc: 90.01%)


Epoch 2/10 [Val]: 100%|██████████| 3498/3498 [11:11<00:00, 5.21it/s]


 Epoch 02 결과 | Train Acc: 89.95% | Val Loss: 0.2025 Val Acc: 90.59%
[추천] [신기록 달성!] 가중치 저장 완료: checkpoints/best_redimnet.pt (Acc: 90.59%)


...

 Epoch 08 결과 | Train Acc: 92.73% | Val Loss: 0.1755 Val Acc: 91.75%
[추천] [신기록 달성!] 가중치 저장 완료: checkpoints/best_redimnet.pt (Acc: 91.75%)


Epoch 9/10 [Val]: 100%|██████████| 3498/3498 [10:39<00:00, 5.47it/s]


 Epoch 09 결과 | Train Acc: 93.07% | Val Loss: 0.1737 Val Acc: 91.89%
[추천] [신기록 달성!] 가중치 저장 완료: checkpoints/best_redimnet.pt (Acc: 91.89%)


Epoch 10/10 [Val]: 100%|██████████| 3498/3498 [10:51<00:00, 5.37it/s]

 Epoch 10 결과 | Train Acc: 93.26% | Val Loss: 0.1783 Val Acc: 91.80%


# ============================================================
# [속도] [Part 2] ECAPA-TDNN 풀학습 파이프라인 (화자 음색 시계열 모델)
# ============================================================
> **특징**: 1D 시간 지연 합성곱(TDNN) + Squeeze-and-Excitation + 통계적 풀링 구조 
> **목적**: ReDimNet(2D Conv + MHA)과 완전히 다른 시계열 음색 특징을 학습하여 앙상블 시너지 극대화 
> **체크포인트 저장 위치**: `checkpoints/best_ecapa_tdnn.pt`

In [7]:
# ============================================================
# ECAPA-TDNN 독립 풀학습 실행 셀
# (ReDimNet의 이전 학습 로그를 건드리지 않고 독립적으로 실행됩니다)
# ============================================================
def train_ecapa_model():
  ecapa_type = 'ecapa_tdnn'
  ecapa_n_mels, ecapa_n_fft, ecapa_hop = 80, 512, 160
  ecapa_epochs = 10
  ecapa_batch_size = 32
  
  print('=' * 65)
  print(f' [RTX 3060 전용] {ecapa_type.upper()} Full Training Pipeline 시작')
  print(f' - Train: {TRAIN_DIR} | Val: {VAL_DIR}')
  print(f' - Batch: {ecapa_batch_size} | Epochs: {ecapa_epochs} | n_mels: {ecapa_n_mels}')
  print(f' - AMP 활성화 | BCEWithLogitsLoss + Random Crop')
  print('=' * 65)
  
  # 1. 데이터셋 & 데이터로더 생성
  train_ds = DCCAudioDataset(TRAIN_DIR, is_train=True, max_files=None, n_mels=ecapa_n_mels, n_fft=ecapa_n_fft, hop_length=ecapa_hop)
  val_ds = DCCAudioDataset(VAL_DIR, is_train=False, max_files=None, n_mels=ecapa_n_mels, n_fft=ecapa_n_fft, hop_length=ecapa_hop)
  
  train_loader = DataLoader(train_ds, batch_size=ecapa_batch_size, shuffle=True, num_workers=0, pin_memory=True)
  val_loader = DataLoader(val_ds, batch_size=ecapa_batch_size, shuffle=False, num_workers=0, pin_memory=True)
  print(f'[확인] 데이터 로드 완료! (Train: {len(train_ds):,} 샘플 | Val: {len(val_ds):,} 샘플)')
  
  # 2. ECAPA-TDNN 모델 생성
  model = build_selected_model('ecapa_tdnn').to(device)
  
  # 3. 벤치마크 사전 가중치 불러오기 (전이학습)
  ckpt_path = 'test_results/checkpoints/best_ecapa_tdnn.pt'
  if os.path.exists(ckpt_path):
    print(f' 저장된 가중치 발견! 불러옵니다: {ckpt_path}')
    try:
      ckpt = torch.load(ckpt_path, map_location=device)
      state_dict = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
      m_dict = model.state_dict()
      matched = {k: v for k, v in state_dict.items() if k in m_dict and v.shape == m_dict[k].shape}
      m_dict.update(matched)
      model.load_state_dict(m_dict)
      print(f'[확인] 가중치 로드 성공! (일치 레이어 {len(matched)}개 복구 완료)')
    except Exception as e:
      print(f'[경고] 가중치 로드 실패: {e}')
  else:
    print(' 이전 가중치가 없어 바닥부터 새로 학습합니다.')
    
  # 4. 손실함수, 옵티마이저, 스케줄러
  criterion = nn.BCEWithLogitsLoss()
  optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
  scheduler = CosineAnnealingLR(optimizer, T_max=ecapa_epochs, eta_min=1e-6)
  scaler = GradScaler()
  best_acc = 0.0
  
  # 5. 에폭 루프
  for epoch in range(1, ecapa_epochs + 1):
    model.train()
    tr_loss, tr_correct, tr_total = 0.0, 0, 0
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{ecapa_epochs} [Train ECAPA]')
    for inputs, labels in pbar:
      inputs, labels = inputs.to(device), labels.to(device).unsqueeze(1)
      optimizer.zero_grad()
      
      with autocast():
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
      scaler.scale(loss).backward()
      scaler.step(optimizer)
      scaler.update()
      
      tr_loss += loss.item() * inputs.size(0)
      preds = (torch.sigmoid(outputs) >= 0.5).float()
      tr_total += labels.size(0)
      tr_correct += preds.eq(labels).sum().item()
      pbar.set_postfix({'loss': f'{loss.item():.4f}'})
      
    scheduler.step()
    tr_acc = tr_correct / tr_total
    
    # Validation
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
      for inputs, labels in tqdm(val_loader, desc=f'Epoch {epoch}/{ecapa_epochs} [Val ECAPA]'):
        inputs, labels = inputs.to(device), labels.to(device).unsqueeze(1)
        with autocast():
          outputs = model(inputs)
          loss = criterion(outputs, labels)
        val_loss += loss.item() * inputs.size(0)
        preds = (torch.sigmoid(outputs) >= 0.5).float()
        val_total += labels.size(0)
        val_correct += preds.eq(labels).sum().item()
        
    val_acc = val_correct / val_total
    val_loss = val_loss / val_total
    
    print(f' [ECAPA-TDNN] Epoch {epoch:02d} | Train Acc: {tr_acc*100:.2f}% | Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%')
    if val_acc > best_acc:
      best_acc = val_acc
      save_path = 'checkpoints/best_ecapa_tdnn.pt'
      torch.save(model.state_dict(), save_path)
      print(f'[추천] [ECAPA 신기록!] 저장 완료: {save_path} (Acc: {best_acc*100:.2f}%)')

# ECAPA-TDNN 풀학습 실행
train_ecapa_model()


 [RTX 3060 전용] ECAPA_TDNN Full Training Pipeline 시작
 - Train: ../data/train | Val: ../data/val
 - Batch: 32 | Epochs: 10 | n_mels: 80
 - AMP 활성화 | BCEWithLogitsLoss + Random Crop
[확인] 데이터 로드 완료! (Train: 873,137 샘플 | Val: 111,919 샘플)
 저장된 가중치 발견! 불러옵니다: test_results/checkpoints/best_ecapa_tdnn.pt
[확인] 가중치 로드 성공! (일치 레이어 231개 복구 완료)


Epoch 1/10 [Val ECAPA]: 100%|██████████| 3498/3498 [12:06<00:00, 4.82it/s]


 [ECAPA-TDNN] Epoch 01 | Train Acc: 87.97% | Val Loss: 0.2160 | Val Acc: 90.01%
[추천] [ECAPA 신기록!] 저장 완료: checkpoints/best_ecapa_tdnn.pt (Acc: 90.01%)


Epoch 2/10 [Val ECAPA]: 100%|██████████| 3498/3498 [11:22<00:00, 5.12it/s]


 [ECAPA-TDNN] Epoch 02 | Train Acc: 90.12% | Val Loss: 0.2192 | Val Acc: 89.59%


...

 [ECAPA-TDNN] Epoch 08 | Train Acc: 93.07% | Val Loss: 0.1706 | Val Acc: 92.10%
[추천] [ECAPA 신기록!] 저장 완료: checkpoints/best_ecapa_tdnn.pt (Acc: 92.10%)


Epoch 9/10 [Val ECAPA]: 100%|██████████| 3498/3498 [11:15<00:00, 5.18it/s]


 [ECAPA-TDNN] Epoch 09 | Train Acc: 93.40% | Val Loss: 0.1724 | Val Acc: 92.09%


Epoch 10/10 [Val ECAPA]: 100%|██████████| 3498/3498 [11:59<00:00, 4.86it/s]

 [ECAPA-TDNN] Epoch 10 | Train Acc: 93.69% | Val Loss: 0.1764 | Val Acc: 92.08%


# ============================================================
# [Part 2.5] AudioResNet-50 풀학습 파이프라인 (2D 비전 텍스처 앵커)
# ============================================================
> **특징**: 2D ImageNet 사전학습 가중치 기반 전이학습 (23.50M 파라미터) 
> **음향 설정**: `n_mels=128`, `n_fft=2048`, `hop_length=512` (고해상도 2D 멜 스펙트로그램) 
> **목적**: 1차 90.20% 베이스라인을 풀데이터(87만 샘플)로 재학습하여 91%+ 로 도약 및 앙상블 앵커 확보 
> **체크포인트 저장 위치**: `checkpoints/best_resnet50.pt`

In [8]:
# ============================================================
# AudioResNet-50 독립 풀학습 실행 셀
# (ReDimNet과 ECAPA의 학습 로그를 전혀 건드리지 않고 독립적으로 실행됩니다)
# ============================================================
def train_resnet_model():
  resnet_type = 'resnet50'
  resnet_n_mels, resnet_n_fft, resnet_hop = 128, 2048, 512
  resnet_epochs = 10
  resnet_batch_size = 32
  
  print('=' * 65)
  print(f' [RTX 3060 전용] AudioResNet-50 Full Training Pipeline 시작')
  print(f' - Train: {TRAIN_DIR} | Val: {VAL_DIR}')
  print(f' - Batch: {resnet_batch_size} | Epochs: {resnet_epochs} | n_mels: {resnet_n_mels}')
  print(f' - AMP 활성화 | BCEWithLogitsLoss + Random Crop')
  print('=' * 65)
  
  # 1. 데이터셋 & 데이터로더 생성 (ResNet 전용 n_mels=128 고해상도)
  train_ds = DCCAudioDataset(TRAIN_DIR, is_train=True, max_files=None, n_mels=resnet_n_mels, n_fft=resnet_n_fft, hop_length=resnet_hop)
  val_ds = DCCAudioDataset(VAL_DIR, is_train=False, max_files=None, n_mels=resnet_n_mels, n_fft=resnet_n_fft, hop_length=resnet_hop)
  
  train_loader = DataLoader(train_ds, batch_size=resnet_batch_size, shuffle=True, num_workers=0, pin_memory=True)
  val_loader = DataLoader(val_ds, batch_size=resnet_batch_size, shuffle=False, num_workers=0, pin_memory=True)
  print(f'[확인] 데이터 로드 완료! (Train: {len(train_ds):,} 샘플 | Val: {len(val_ds):,} 샘플)')
  
  # 2. ResNet-50 모델 생성
  model = build_selected_model('resnet50').to(device)
  
  # 3. 기존 90% 베이스라인 가중치 불러오기 (전이학습)
  ckpt_candidates = [
    'results_benchmark/checkpoints/baseline_resnet50_90.pt',
    'test_results/checkpoints/best_resnet50.pt',
    'results/checkpoints/best_resnet50.pt'
  ]
  loaded = False
  for ckpt_path in ckpt_candidates:
    if os.path.exists(ckpt_path):
      print(f' 저장된 ResNet 가중치 발견! 불러옵니다: {ckpt_path}')
      try:
        ckpt = torch.load(ckpt_path, map_location=device)
        state_dict = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
        m_dict = model.state_dict()
        matched = {k: v for k, v in state_dict.items() if k in m_dict and v.shape == m_dict[k].shape}
        m_dict.update(matched)
        model.load_state_dict(m_dict)
        print(f'[확인] ResNet 가중치 로드 성공! (일치 레이어 {len(matched)}개 복구 완료)')
        loaded = True
        break
      except Exception as e:
        print(f'[경고] 가중치 로드 실패: {e}')
  if not loaded:
    print(' 이전 가중치가 없어 ImageNet 사전학습 가중치 기반으로 학습합니다.')
    
  # 4. 손실함수, 옵티마이저, 스케줄러
  criterion = nn.BCEWithLogitsLoss()
  optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
  scheduler = CosineAnnealingLR(optimizer, T_max=resnet_epochs, eta_min=1e-6)
  scaler = GradScaler()
  best_acc = 0.0
  
  # 5. 에폭 루프
  for epoch in range(1, resnet_epochs + 1):
    model.train()
    tr_loss, tr_correct, tr_total = 0.0, 0, 0
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{resnet_epochs} [Train ResNet]')
    for inputs, labels in pbar:
      inputs, labels = inputs.to(device), labels.to(device).unsqueeze(1)
      optimizer.zero_grad()
      
      with autocast():
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
      scaler.scale(loss).backward()
      scaler.step(optimizer)
      scaler.update()
      
      tr_loss += loss.item() * inputs.size(0)
      preds = (torch.sigmoid(outputs) >= 0.5).float()
      tr_total += labels.size(0)
      tr_correct += preds.eq(labels).sum().item()
      pbar.set_postfix({'loss': f'{loss.item():.4f}'})
      
    scheduler.step()
    tr_acc = tr_correct / tr_total
    
    # Validation
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
      for inputs, labels in tqdm(val_loader, desc=f'Epoch {epoch}/{resnet_epochs} [Val ResNet]'):
        inputs, labels = inputs.to(device), labels.to(device).unsqueeze(1)
        with autocast():
          outputs = model(inputs)
          loss = criterion(outputs, labels)
        val_loss += loss.item() * inputs.size(0)
        preds = (torch.sigmoid(outputs) >= 0.5).float()
        val_total += labels.size(0)
        val_correct += preds.eq(labels).sum().item()
        
    val_acc = val_correct / val_total
    val_loss = val_loss / val_total
    
    print(f' [ResNet-50] Epoch {epoch:02d} | Train Acc: {tr_acc*100:.2f}% | Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%')
    if val_acc > best_acc:
      best_acc = val_acc
      save_path = 'checkpoints/best_resnet50.pt'
      torch.save(model.state_dict(), save_path)
      print(f'[추천] [ResNet 신기록!] 저장 완료: {save_path} (Acc: {best_acc*100:.2f}%)')

# AudioResNet-50 풀학습 실행
train_resnet_model()


 [RTX 3060 전용] AudioResNet-50 Full Training Pipeline 시작
 - Train: ../data/train | Val: ../data/val
 - Batch: 32 | Epochs: 10 | n_mels: 128
 - AMP 활성화 | BCEWithLogitsLoss + Random Crop
[확인] 데이터 로드 완료! (Train: 873,137 샘플 | Val: 111,919 샘플)
 저장된 ResNet 가중치 발견! 불러옵니다: results_benchmark/checkpoints/baseline_resnet50_90.pt
[확인] ResNet 가중치 로드 성공! (일치 레이어 0개 복구 완료)


Epoch 1/10 [Val ResNet]: 100%|██████████| 3498/3498 [14:02<00:00, 4.15it/s]


 [ResNet-50] Epoch 01 | Train Acc: 88.56% | Val Loss: 0.2159 | Val Acc: 89.90%
[추천] [ResNet 신기록!] 저장 완료: checkpoints/best_resnet50.pt (Acc: 89.90%)


Epoch 2/10 [Val ResNet]: 100%|██████████| 3498/3498 [13:22<00:00, 4.36it/s]


 [ResNet-50] Epoch 02 | Train Acc: 90.75% | Val Loss: 0.2004 | Val Acc: 90.61%
[추천] [ResNet 신기록!] 저장 완료: checkpoints/best_resnet50.pt (Acc: 90.61%)


...

 [ResNet-50] Epoch 08 | Train Acc: 94.80% | Val Loss: 0.2737 | Val Acc: 91.14%


Epoch 9/10 [Val ResNet]: 100%|██████████| 3498/3498 [12:21<00:00, 4.72it/s]


 [ResNet-50] Epoch 09 | Train Acc: 95.02% | Val Loss: 0.3271 | Val Acc: 91.19%
[추천] [ResNet 신기록!] 저장 완료: checkpoints/best_resnet50.pt (Acc: 91.19%)


Epoch 10/10 [Val ResNet]: 100%|██████████| 3498/3498 [12:46<00:00, 4.56it/s]


 [ResNet-50] Epoch 10 | Train Acc: 95.12% | Val Loss: 0.3656 | Val Acc: 91.20%
[추천] [ResNet 신기록!] 저장 완료: checkpoints/best_resnet50.pt (Acc: 91.20%)


# [Part 3] 합성 입력 모델 연산 벤치마크

무작위 tensor로 모델 forward 지연시간과 batch 32 처리량을 측정한다. 시간축은 모두 300프레임이며 오디오 읽기·리샘플링·Mel 계산은 포함하지 않는다. checkpoint를 로드하지 않은 아키텍처 벤치마크이고 정확도는 기존 학습 성적 상수다. ResNet 실제 3초 입력은 94프레임이므로 종단간 속도나 실시간성 평가로 해석하지 않는다. 파일 크기 단위는 MiB다.


In [7]:
import time
import pandas as pd

print(' [합성 입력 forward 벤치마크 시작]')

benchmark_targets = [
  ('ReDimNet2-B2', ReDimNet2_B2(num_classes=1), (1, 80, 300), 'checkpoints/best_redimnet.pt', 91.89),
  ('ECAPA-TDNN', ECAPA_TDNN(in_channels=80, channels=512, num_classes=1), (1, 80, 300), 'checkpoints/best_ecapa_tdnn.pt', 92.10),
  ('AudioResNet-50', get_audio_resnet50(), (1, 1, 128, 300), 'checkpoints/best_resnet50.pt', 91.20)
]

bench_rows = []
for name, model, in_shape, ckpt_p, known_acc in benchmark_targets:
  model = model.to(device)
  model.eval()
  
  # 1. 파라미터 & 파일 크기
  params_m = sum(p.numel() for p in model.parameters()) / 1e6
  size_mb = (os.path.getsize(ckpt_p) / (1024*1024)) if os.path.exists(ckpt_p) else 0.0
  
  # 2. Warmup & Latency 측정 (200회 반복)
  dummy = torch.randn(*in_shape, device=device)
  with torch.no_grad():
    for _ in range(30):
      _ = model(dummy)
  if device.type == 'cuda':
    torch.cuda.synchronize()
    
  t0 = time.perf_counter()
  with torch.no_grad():
    for _ in range(200):
      _ = model(dummy)
      if device.type == 'cuda':
        torch.cuda.synchronize()
  t1 = time.perf_counter()
  lat_ms = ((t1 - t0) / 200) * 1000
  
  # 3. Throughput 측정 (batch=32, 50회)
  batch_shape = (32,) + in_shape[1:]
  dummy_b = torch.randn(*batch_shape, device=device)
  t0 = time.perf_counter()
  with torch.no_grad():
    for _ in range(50):
      _ = model(dummy_b)
      if device.type == 'cuda':
        torch.cuda.synchronize()
  t1 = time.perf_counter()
  throughput = (32 * 50) / (t1 - t0)
  
  bench_rows.append({
    '모델명': name,
    '파라미터': f'{params_m:.2f} M',
    '파일크기': f'{size_mb:.1f} MiB',
    '지연시간': f'{lat_ms:.2f} ms',
    '초당 처리량': f'{throughput:.1f} 건/초',
    '최고 정확도': f'{known_acc:.2f}%' if known_acc else '학습 후 반영'
  })

df_bench = pd.DataFrame(bench_rows)
print('\n' + '=' * 80)
print(' [현재 실행 장치의 합성 입력] 모델 연산 비교표')
print('=' * 80)
display(df_bench)


 [효율성 정밀 실측 벤치마크 시작 (RTX 3060 기준)]

 [RTX 3060 실측] 모델별 다차원 효율성 비교표


,모델명,파라미터,파일크기,지연시간,RTF(실시간성),초당 처리량,최고 정확도
0,ReDimNet2-B2,2.57 M,9.8 MB,1.73 ms,0.0006,1999.4 건/초,91.89%
1,ECAPA-TDNN,5.80 M,22.2 MB,5.12 ms,0.0017,1304.2 건/초,92.10%
2,AudioResNet-50,23.50 M,90.0 MB,4.65 ms,0.0016,632.9 건/초,91.20%


# [Part 4] 초기 부분 검증 데이터 로딩

최종 평가 전에 작성한 부분 검증 준비 셀이다. 아래 셀은 데이터만 로드하며 최종 성적을 계산하지 않는다. 최종 균등 평균과 임계값 0.50 평가는 Part 6을 사용한다.


In [7]:
from sklearn.metrics import accuracy_score, f1_score

print(' [3대장 Soft Voting 앙상블 검증 시작]')

# 가중치 파일 존재 확인
w_resnet = 'checkpoints/best_resnet50.pt' if os.path.exists('checkpoints/best_resnet50.pt') else 'results_benchmark/checkpoints/baseline_resnet50_90.pt'
w_redim = 'checkpoints/best_redimnet.pt'
w_ecapa = 'checkpoints/best_ecapa_tdnn.pt' if os.path.exists('checkpoints/best_ecapa_tdnn.pt') else 'test_results/checkpoints/best_ecapa_tdnn.pt'

print(f'1. ResNet-50: {w_resnet} (존재: {os.path.exists(w_resnet)})')
print(f'2. ReDimNet:  {w_redim} (존재: {os.path.exists(w_redim)})')
print(f'3. ECAPA-TDNN: {w_ecapa} (존재: {os.path.exists(w_ecapa)})')

# 검증 데이터셋 로드 (평가용 5,000개 샘플 빠른 평가)
eval_ds_80 = DCCAudioDataset(VAL_DIR, is_train=False, max_files=50, n_mels=80, n_fft=512, hop_length=160)
eval_ds_128 = DCCAudioDataset(VAL_DIR, is_train=False, max_files=50, n_mels=128, n_fft=2048, hop_length=512)
print(f'[확인] 앙상블 평가용 검증 샘플 로드 완료: {len(eval_ds_80):,}개')


 [3대장 Soft Voting 앙상블 검증 시작]
1. ResNet-50: checkpoints/best_resnet50.pt (존재: True)
2. ReDimNet:  checkpoints/best_redimnet.pt (존재: True)
3. ECAPA-TDNN: checkpoints/best_ecapa_tdnn.pt (존재: True)
[확인] 앙상블 평가용 검증 샘플 로드 완료: 1,553개


# [Part 5] 특화 전처리 v2 미세 조정

v1은 Mel fmin/fmax를 바꿨고, v2는 기본 Mel 설정을 유지하면서 파형에 200~4,000Hz Butterworth bandpass와 filtfilt를 적용한다. v1의 성능 하락 원인을 확정한 것은 아니다. 별도 RMS scaling, Delta 3채널, VAD는 구현하지 않았다. Git 2c6cb4a의 3에폭 완료 결과와 이후 추가 실행의 중단 결과는 마지막 요약과 역사 노트북을 참고한다.


In [8]:
# ============================================================
# ️ 도메인 특화 전처리 전용 Dataset v2 (Mel-Bin 왜곡 제거형)
# (fmin, fmax 조작 대신 원본 파형에 직접 200~4000Hz 디지털 밴드패스 필터 적용)
# ============================================================
import scipy.signal

class DCCSpecializedAudioDataset(Dataset):
  def __init__(self, data_dir, is_train=True, max_files=None, n_mels=80, n_fft=512, hop_length=160,
         fmin=200, fmax=4000, window_sec=3.0):
    self.data_dir = data_dir
    self.is_train = is_train
    self.n_mels = n_mels
    self.n_fft = n_fft
    self.hop_length = hop_length
    self.fmin = fmin
    self.fmax = fmax
    self.window_sec = window_sec
    self.sr = 16000
    self.target_samples = int(self.sr * self.window_sec)
    
    # 밴드패스 필터 설계 (Butterworth)
    nyq = 0.5 * self.sr
    low = self.fmin / nyq
    high = self.fmax / nyq
    self.b, self.a = scipy.signal.butter(4, [low, high], btype='band')
    
    self.samples = self._parse_data(max_files)
    
  def _parse_data(self, max_files):
    samples = []
    label_dir = os.path.join(self.data_dir, 'label')
    audio_dir = os.path.join(self.data_dir, 'audio')
    if not os.path.exists(label_dir):
      label_dir = self.data_dir
      audio_dir = self.data_dir
      
    json_files = glob.glob(os.path.join(label_dir, '**', '*.json'), recursive=True)
    if max_files:
      json_files = json_files[:max_files]
      
    for jf in json_files:
      with open(jf, 'r', encoding='utf-8') as f:
        data = json.load(f)
      wav_name = os.path.basename(jf).replace('.json', '.wav')
      wav_path = os.path.join(audio_dir, wav_name)
      if not os.path.exists(wav_path):
        wav_search = glob.glob(os.path.join(audio_dir, '**', wav_name), recursive=True)
        if wav_search:
          wav_path = wav_search[0]
        else:
          continue
          
      if 'utterances' in data:
        for utt in data['utterances']:
          samples.append({
            'wav_path': wav_path,
            'start_ms': float(utt.get('startAt', 0)),
            'end_ms': float(utt.get('endAt', 0)),
            'label': int(utt.get('speaker', 0))
          })
    return samples
    
  def __len__(self):
    return len(self.samples)
    
  def __getitem__(self, idx):
    sample = self.samples[idx]
    wav_path = sample['wav_path']
    try:
      start_sec = sample['start_ms'] / 1000.0
      end_sec = sample['end_ms'] / 1000.0
      clip_dur = max(0.01, end_sec - start_sec)
      y, sr = librosa.load(wav_path, sr=self.sr, offset=start_sec, duration=clip_dur)
      
      # [핵심 수정 1] Melspectrogram의 fmin/fmax 조작 대신, 원본 파형에 물리적 대역 필터 적용!
      # 이를 통해 Mel-Bin 주파수 왜곡 없이 오직 200~4000Hz 소리만 순수하게 추출합니다.
      if len(y) > 33: # 필터 최소 길이 보장
        y = scipy.signal.filtfilt(self.b, self.a, y).astype(np.float32)
    except Exception:
      y = np.zeros(self.target_samples, dtype=np.float32)
      
    cur_len = len(y)
    if cur_len < self.target_samples:
      y = np.pad(y, (0, self.target_samples - cur_len), mode='constant')
    elif cur_len > self.target_samples:
      start_idx = random.randint(0, cur_len - self.target_samples) if self.is_train else (cur_len - self.target_samples) // 2
      y = y[start_idx : start_idx + self.target_samples]
      
    if len(y) < self.n_fft:
      y = np.pad(y, (0, self.n_fft - len(y)), mode='constant')
      
    # [핵심 수정 2] fmin, fmax 파라미터 완전 제거 -> 순정 가중치의 필터 배치를 100% 동일하게 유지
    mel = librosa.feature.melspectrogram(
      y=y, sr=self.sr, n_fft=self.n_fft, hop_length=self.hop_length,
      n_mels=self.n_mels
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    
    # 순정 모델과 완벽하게 동일한 Min-Max 스케일링 적용 (조건 일치화)
    mel_norm = np.clip((mel_db + 80.0) / 80.0, 0.0, 1.0)
    
    tensor_x = torch.tensor(mel_norm, dtype=torch.float32).unsqueeze(0)
    tensor_y = torch.tensor(sample['label'], dtype=torch.float32)
    return tensor_x, tensor_y

print('[확인] 특화 전처리 Dataset v2 클래스(DCCSpecializedAudioDataset) 준비 완료! (Mel-Bin 왜곡 해결)')


[확인] 특화 전처리 Dataset v2 클래스(DCCSpecializedAudioDataset) 준비 완료! (Mel-Bin 왜곡 해결)


In [9]:
# ============================================================
# 특화 전처리 고속 파인튜닝 실행 엔진 함수
# ============================================================
def finetune_specialized(model_name, epochs=3, lr=5e-5, batch_size=32):
  if model_name == 'resnet50':
    n_mels, n_fft, hop = 128, 2048, 512
    base_ckpt = 'checkpoints/best_resnet50.pt'
    known_base_acc = 91.20
  elif model_name == 'redimnet':
    n_mels, n_fft, hop = 80, 512, 160
    base_ckpt = 'checkpoints/best_redimnet.pt'
    known_base_acc = 91.89
  elif model_name == 'ecapa_tdnn':
    n_mels, n_fft, hop = 80, 512, 160
    base_ckpt = 'checkpoints/best_ecapa_tdnn.pt'
    known_base_acc = 92.10
  else:
    raise ValueError(f'Unknown model: {model_name}')
    
  print('=' * 70)
  print(f' [{model_name.upper()}] 도메인 특화 전처리(200~4000Hz) {epochs}에폭 고속 파인튜닝')
  print(f' - 기존 순정 최고 기준점: {known_base_acc:.2f}%')
  print(f' - 불러올 챔피언 가중치: {base_ckpt}')
  print(f' - 파인튜닝 러닝레이트: {lr} (부드러운 적응 모드)')
  print('=' * 70)
  
  # 1. 특화 전처리 데이터셋 생성
  train_ds = DCCSpecializedAudioDataset(TRAIN_DIR, is_train=True, max_files=None, n_mels=n_mels, n_fft=n_fft, hop_length=hop, fmin=200, fmax=4000)
  val_ds = DCCSpecializedAudioDataset(VAL_DIR, is_train=False, max_files=None, n_mels=n_mels, n_fft=n_fft, hop_length=hop, fmin=200, fmax=4000)
  
  train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
  val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
  
  # 2. 모델 생성 및 챔피언 가중치 로드
  model = build_selected_model(model_name).to(device)
  if os.path.exists(base_ckpt):
    ckpt = torch.load(base_ckpt, map_location=device)
    state_dict = ckpt['model_state_dict'] if isinstance(ckpt, dict) and 'model_state_dict' in ckpt else ckpt
    m_dict = model.state_dict()
    matched = {k: v for k, v in state_dict.items() if k in m_dict and v.shape == m_dict[k].shape}
    m_dict.update(matched)
    model.load_state_dict(m_dict)
    print(f' 기존 챔피언 가중치 복원 완료! ({len(matched)}개 레이어 완벽 매칭)')
  else:
    print(f'[경고] 경고: {base_ckpt}를 찾지 못했습니다.')
    
  # 3. 파인튜닝 옵티마이저 & 스케줄러 (낮은 LR로 정밀 튜닝)
  criterion = nn.BCEWithLogitsLoss()
  optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
  scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
  scaler = GradScaler()
  best_acc = float("-inf")
  specialized_save_path = f'checkpoints/best_{model_name}_specialized.pt'
  
  # 4. 고속 파인튜닝 루프
  for epoch in range(1, epochs + 1):
    model.train()
    tr_loss, tr_correct, tr_total = 0.0, 0, 0
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{epochs} [FT {model_name}]')
    for inputs, labels in pbar:
      inputs, labels = inputs.to(device), labels.to(device).unsqueeze(1)
      optimizer.zero_grad()
      
      with autocast():
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
      scaler.scale(loss).backward()
      scaler.step(optimizer)
      scaler.update()
      
      tr_loss += loss.item() * inputs.size(0)
      preds = (torch.sigmoid(outputs) >= 0.5).float()
      tr_total += labels.size(0)
      tr_correct += preds.eq(labels).sum().item()
      pbar.set_postfix({'loss': f'{loss.item():.4f}'})
      
    scheduler.step()
    tr_acc = tr_correct / tr_total
    
    # Validation
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
      for inputs, labels in tqdm(val_loader, desc=f'Epoch {epoch}/{epochs} [Val FT]'):
        inputs, labels = inputs.to(device), labels.to(device).unsqueeze(1)
        with autocast():
          outputs = model(inputs)
          loss = criterion(outputs, labels)
        val_loss += loss.item() * inputs.size(0)
        preds = (torch.sigmoid(outputs) >= 0.5).float()
        val_total += labels.size(0)
        val_correct += preds.eq(labels).sum().item()
        
    val_acc = val_correct / val_total
    val_loss = val_loss / val_total
    
    diff = (val_acc * 100.0) - known_base_acc
    sign = '+' if diff >= 0 else ''
    print(f' [특화 FT {epoch}ep] Val Acc: {val_acc*100:.2f}% (순정 대비 {sign}{diff:.2f}%p) | Val Loss: {val_loss:.4f}')
    
    new_best = val_acc > best_acc
    best_acc = max(best_acc, val_acc)
    if new_best and val_acc > known_base_acc / 100.0:
      torch.save(model.state_dict(), specialized_save_path)
      print(f' [특화 전처리 신기록 경신!] 저장 완료: {specialized_save_path} (Acc: {best_acc*100:.2f}%)')
      
  print('=' * 70)
  print(f' [{model_name}] 특화 전처리 {epochs}에폭 파인튜닝 완료! 최고 기록: {best_acc*100:.2f}%')
  print('=' * 70)

print('[확인] 특화 전처리 파인튜닝 엔진 함수(finetune_specialized) 준비 완료!')


[확인] 특화 전처리 파인튜닝 엔진 함수(finetune_specialized) 준비 완료!


In [12]:
# ============================================================
# 1호 타자: ReDimNet2-B2 특화 전처리 3에폭 고속 파인튜닝
# (기존 순정 최고점: 91.89% -> 특화 전처리 적용 시 점수 상승량 검증)
# ============================================================
finetune_specialized('redimnet', epochs=10, lr=5e-5)


 [REDIMNET] 도메인 특화 전처리(200~4000Hz) 10에폭 고속 파인튜닝
 - 기존 순정 최고 기준점: 91.89%
 - 불러올 챔피언 가중치: checkpoints/best_redimnet.pt
 - 파인튜닝 러닝레이트: 5e-05 (부드러운 적응 모드)
 기존 챔피언 가중치 복원 완료! (91개 레이어 완벽 매칭)


Epoch 1/10 [Val FT]: 100%|██████████| 3498/3498 [25:22<00:00, 2.30it/s]


 [특화 FT 1ep] Val Acc: 91.25% (순정 대비 -0.64%p) | Val Loss: 0.1795


Epoch 2/10 [Val FT]: 100%|██████████| 3498/3498 [33:29<00:00, 1.74it/s]


 [특화 FT 2ep] Val Acc: 91.42% (순정 대비 -0.47%p) | Val Loss: 0.1793


...

 [특화 FT 7ep] Val Acc: 91.29% (순정 대비 -0.60%p) | Val Loss: 0.2057


Epoch 8/10 [Val FT]: 100%|██████████| 3498/3498 [11:06<00:00, 5.25it/s]


 [특화 FT 8ep] Val Acc: 91.33% (순정 대비 -0.56%p) | Val Loss: 0.2124


Epoch 9/10 [FT redimnet]: 64%|██████▍  | 17553/27286 [1:16:44<42:32, 3.81it/s, loss=0.1112] 


KeyboardInterrupt: 

In [ ]:
# ============================================================
# 2호 타자: ECAPA-TDNN 특화 전처리 3에폭 고속 파인튜닝
# (기존 순정 최고점: 92.10% -> 특화 전처리 적용 시 점수 상승량 검증)
# ============================================================
finetune_specialized('ecapa_tdnn', epochs=10, lr=5e-5)


 [ECAPA_TDNN] 도메인 특화 전처리(200~4000Hz) 3에폭 고속 파인튜닝
 - 기존 순정 최고 기준점: 92.10%
 - 불러올 챔피언 가중치: checkpoints/best_ecapa_tdnn.pt
 - 파인튜닝 러닝레이트: 5e-05 (부드러운 적응 모드)


KeyboardInterrupt: 

In [ ]:
# ============================================================
# 3호 타자: AudioResNet-50 특화 전처리 3에폭 고속 파인튜닝
# (기존 순정 최고점: 91.20% -> 특화 전처리 적용 시 점수 상승량 검증)
# ============================================================
finetune_specialized('resnet50', epochs=3, lr=5e-5)


 [RESNET50] 도메인 특화 전처리(200~4000Hz) 3에폭 고속 파인튜닝
 - 기존 순정 최고 기준점: 91.20%
 - 불러올 챔피언 가중치: checkpoints/best_resnet50.pt
 - 파인튜닝 러닝레이트: 5e-05 (부드러운 적응 모드)
 기존 챔피언 가중치 복원 완료! (320개 레이어 완벽 매칭)


Epoch 1/3 [Val FT]: 100%|██████████| 3498/3498 [11:52<00:00, 4.91it/s]


 [특화 FT 1ep] Val Acc: 90.54% (순정 대비 -0.66%p) | Val Loss: 0.2034


Epoch 2/3 [Val FT]: 100%|██████████| 3498/3498 [12:08<00:00, 4.80it/s]


 [특화 FT 2ep] Val Acc: 90.67% (순정 대비 -0.53%p) | Val Loss: 0.2136


Epoch 3/3 [Val FT]: 100%|██████████| 3498/3498 [11:30<00:00, 5.07it/s]

 [특화 FT 3ep] Val Acc: 90.70% (순정 대비 -0.50%p) | Val Loss: 0.2634
 [resnet50] 특화 전처리 3에폭 파인튜닝 완료! 최고 기록: 91.20%


# [Part 6] 최종 균등 평균 앙상블과 임계값 0.50

각 기본 모델의 신고자 1 확률을 1/3씩 평균하고, 평균 확률이 0.50 이상이면 1로 분류한다. 현재 최종 선택은 이 설정이다. 아래 출력은 Git 3638dab의 기존 실행 기록이며, 변경된 코드의 새 실행 결과가 아니다.


In [13]:
# ============================================================
# [Part 6] 기본 모델 균등 평균 앙상블과 임계값 0.50 평가
# (기본 최고 가중치만 사용하고 strict loading 적용)
# ============================================================
import os
import torch
import numpy as np
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast
from sklearn.metrics import accuracy_score, f1_score, classification_report

# 0. 평가 범위 설정 (빠른 검증: 100개 파일 (~1.2만 샘플) | 전체 검증: None (11.2만 샘플))
EVAL_MAX_FILES = None

# 1. 기본 최고 가중치 선택 (최종 설정)
def pick_best_weight(model_name):
  base_path = f'checkpoints/best_{model_name}.pt'
  if not os.path.exists(base_path):
    raise FileNotFoundError(base_path)
  print(f'[{model_name}] 기본 최고 가중치: {base_path}')
  return base_path, False

print('=' * 65)
print(' [가중치 탐색 및 자동 로드]')
w_redim, redim_is_spec = pick_best_weight('redimnet')
w_ecapa, ecapa_is_spec = pick_best_weight('ecapa_tdnn')
w_resnet, resnet_is_spec = pick_best_weight('resnet50')
print('=' * 65)

# 2. 모델 로드 함수
def load_trained_model(model_name, weight_path):
  from pathlib import Path
  from m2.infer import load_model_weights
  model = build_selected_model(model_name).to(device)
  return load_model_weights(model, Path(weight_path), device)

m_redim = load_trained_model('redimnet', w_redim)
m_ecapa = load_trained_model('ecapa_tdnn', w_ecapa)
m_resnet = load_trained_model('resnet50', w_resnet)

# 3. 모델별 맞춤 평가 Dataset 생성 (특화 모델은 특화 전처리, 순정 모델은 순정 전처리)
def get_eval_dataset(model_name, is_spec, max_files):
  if model_name == 'resnet50':
    n_mels, n_fft, hop = 128, 2048, 512
  else:
    n_mels, n_fft, hop = 80, 512, 160
  
  if is_spec:
    return DCCSpecializedAudioDataset(VAL_DIR, is_train=False, max_files=max_files,
                     n_mels=n_mels, n_fft=n_fft, hop_length=hop, fmin=200, fmax=4000)
  else:
    return DCCAudioDataset(VAL_DIR, is_train=False, max_files=max_files,
               n_mels=n_mels, n_fft=n_fft, hop_length=hop)

print(f'\n 검증 데이터셋 로드 중... (EVAL_MAX_FILES={EVAL_MAX_FILES})')
ds_r = get_eval_dataset('redimnet', redim_is_spec, EVAL_MAX_FILES)
ds_e = get_eval_dataset('ecapa_tdnn', ecapa_is_spec, EVAL_MAX_FILES)
ds_res = get_eval_dataset('resnet50', resnet_is_spec, EVAL_MAX_FILES)

assert ds_r.samples == ds_e.samples == ds_res.samples, '앙상블 발화 순서 불일치'
loader_r = DataLoader(ds_r, batch_size=32, shuffle=False, num_workers=0)
loader_e = DataLoader(ds_e, batch_size=32, shuffle=False, num_workers=0)
loader_res = DataLoader(ds_res, batch_size=32, shuffle=False, num_workers=0)
print(f'[확인] 평가 샘플 수: {len(ds_r):,}개')

# 4. 3대 모델 동시 추론 및 확률값(Sigmoid) 수집
probs_r, probs_e, probs_res, all_labels = [], [], [], []

print(' 3대 모델 앙상블 동시 추론 진행 중...')
with torch.no_grad():
  for (xr, yr), (xe, _), (xres, _) in zip(loader_r, loader_e, loader_res):
    xr, xe, xres = xr.to(device), xe.to(device), xres.to(device)
    all_labels.extend(yr.numpy())
    
    with autocast():
      pr = torch.sigmoid(m_redim(xr)).cpu().numpy().flatten()
      pe = torch.sigmoid(m_ecapa(xe)).cpu().numpy().flatten()
      pres = torch.sigmoid(m_resnet(xres)).cpu().numpy().flatten()
      
    probs_r.extend(pr)
    probs_e.extend(pe)
    probs_res.extend(pres)

y_true = np.array(all_labels)
p_r = np.array(probs_r)
p_e = np.array(probs_e)
p_res = np.array(probs_res)

# 5. 단일 모델별 개별 성능 리포트 (기본 threshold 0.5)
print('\n' + '=' * 65)
print(' [단일 모델별 성능 비교 (Cut-off 0.50 고정)]')
print(f' 1. ReDimNet2-B2: 정확도 {accuracy_score(y_true, p_r >= 0.5)*100:.2f}% | Macro F1: {f1_score(y_true, p_r >= 0.5, average="macro"):.4f}')
print(f' 2. ECAPA-TDNN:  정확도 {accuracy_score(y_true, p_e >= 0.5)*100:.2f}% | Macro F1: {f1_score(y_true, p_e >= 0.5, average="macro"):.4f}')
print(f' 3. AudioResNet-50: 정확도 {accuracy_score(y_true, p_res >= 0.5)*100:.2f}% | Macro F1: {f1_score(y_true, p_res >= 0.5, average="macro"):.4f}')
print('=' * 65)

# 6. 최종 설정: 사전 정의 균등 가중치 Soft Voting 앙상블 (Simple Average, 1/3씩)
print('\n' + '=' * 65)
print(' [최종 설정: 사전 균등 가중치(1/3씩) Soft Voting 앙상블]')
p_ensemble = (p_e + p_r + p_res) / 3.0

# 7. 최종 선택: 결정 임계값 0.50 고정
best_th = 0.50
final_preds = (p_ensemble >= best_th).astype(int)
final_acc = accuracy_score(y_true, final_preds)
best_f1 = f1_score(y_true, final_preds, average='macro')

print('\n' + '' * 32)
print(' [3대장 Soft Voting 앙상블 최종 검증 결과 (균등 평균, 임계값 0.50)]')
print(f' - 적용 분류 임계값 (Cut-off): {best_th:.2f} (최종 설정 0.50 고정)')
print(f' - 앙상블 최종 검증 정확도 (Accuracy): {final_acc*100:.2f}%')
print(f' - 앙상블 최종 Macro F1-Score:    {best_f1:.4f}')
print('' * 32)
print('\n[정식 세부 분류 성적표 (Classification Report)]')
print(classification_report(y_true, final_preds, labels=[0, 1], target_names=['상황실 (Dispatcher, 0)', '신고자 (Caller, 1)'], zero_division=0, digits=4))

# Preserve exact predictions and sample identities for future independent verification.
from pathlib import Path
audit_dir = Path('reports')
audit_dir.mkdir(exist_ok=True)
np.savez_compressed(audit_dir / 'validation_predictions.npz',
  y_true=y_true, p_redim=p_r, p_ecapa=p_e, p_resnet=p_res,
  p_ensemble=p_ensemble, predictions=final_preds)
with (audit_dir / 'validation_samples.json').open('w', encoding='utf-8') as f:
  json.dump(ds_r.samples, f, ensure_ascii=False, indent=2)


 [가중치 탐색 및 자동 로드]
 [redimnet ] 순정 최고 가중치 적용:  checkpoints/best_redimnet.pt
 [ecapa_tdnn] 순정 최고 가중치 적용:  checkpoints/best_ecapa_tdnn.pt
 [resnet50 ] 순정 최고 가중치 적용:  checkpoints/best_resnet50.pt

 검증 데이터셋 로드 중... (EVAL_MAX_FILES=None)
[확인] 평가 샘플 수: 111,919개
 3대 모델 앙상블 동시 추론 진행 중...

 [단일 모델별 성능 비교 (Cut-off 0.50 고정)]
 1. ReDimNet2-B2: 정확도 91.89% | Macro F1: 0.9185
 2. ECAPA-TDNN:  정확도 92.10% | Macro F1: 0.9206
 3. AudioResNet-50: 정확도 91.20% | Macro F1: 0.9115

 [대회 규정 준수: 사전 균등 가중치(1/3씩) Soft Voting 앙상블]


 [3대장 Soft Voting 앙상블 최종 검증 결과 (대회 규정 100% 준수)]
 - 적용 분류 임계값 (Cut-off): 0.50 (대회 규정 0.50 고정)
 - 앙상블 최종 검증 정확도 (Accuracy): 92.48%
 - 앙상블 최종 Macro F1-Score:    0.9244


[정식 세부 분류 성적표 (Classification Report)]
         precision  recall f1-score  support

  신고자 (Caller)   0.9495  0.8912  0.9194   53830
상황실 (Dispatcher)   0.9042  0.9560  0.9294   58089

    accuracy             0.9248  111919
    macro avg   0.9268  0.9236  0.9244  111919
  weighted avg   0.9260  0.9248  0.9246  111919


## 기존 기록의 최종 결과 표

아래는 보존된 로그를 숫자 라벨 기준으로 정리한 표이며 재실행 결과가 아니다.

| 항목 | 정확도 | Macro F1 | 검증 발화 수 |
|---|---|---|---|
| ReDimNet 계열 로컬 구현 | 91.89% | 0.9185 | 111,919 |
| ECAPA-TDNN | 92.10% | 0.9206 | 111,919 |
| AudioResNet-50 | 91.20% | 0.9115 | 111,919 |
| 균등 평균 앙상블 임계값 0.50 | 92.48% | 0.9244 | 111,919 |

밴드패스 실험은 ReDimNet 8개 완료 에폭 중 5에폭에서 91.58%, ResNet 3개
완료 에폭 중 3에폭에서 90.70%이다. ECAPA는 Git 2c6cb4a의 3에폭 완료 로그에서 91.82%를 기록했다. 이전 ReDim 3에폭 최고는 91.57%였다. 세 모델의 완료 FT는 모두 기본 성적보다 낮았다.
FT 출력 마지막의 ResNet 91.20%는 초기 최고값 변수 오류이므로 FT 성적이 아니다.
정확한 혼동 행렬과 수정된 추론 경로의 정확도는 원본 데이터로 다시 평가해야 한다.


# [Part 7] 최종 전수 실측 검증 및 7대 후보 비교 (2026-10-08 검증)

2026-10-08 Windows RTX 3060 환경에서 111,919건 전수 모집단(28건 WAV 누락 제외)에 대해 결정 임계값 0.50을 고정하고 7대 후보 모델을 전수 실측 비교한 결과다.

### 1. 7대 후보 모델 전수 실측 결과표 (111,919건, 임계값 0.50 고정)

| 후보 ID | 모델 구성 | 검증 정확도 (Accuracy) | Macro F1 | 정답수 / 전체 | 파라미터 수 | 체크포인트 크기 | 비고 |
|---|---|---|---|---|---|---|---|
| **E-REN** | **3모델 균등 앙상블 (1/3)** | **92.46%** | **0.9242** | **103,484 / 111,919** | 31.87M | 122.04 MiB | **단일/균등 7대 비교군 중 최고 (1위)** |
| **E-RE** | **ReDimNet + ECAPA (2모델)** | **92.42%** | **0.9238** | **103,435 / 111,919** | **8.36M** | **32.08 MiB** | **초경량 실용 최적 대안 (74% 경량화)** |
| S-E | ECAPA-TDNN 단독 | 92.05% | 0.9201 | 103,018 / 111,919 | 5.80M | 22.23 MiB | 단일 백본 1위 |
| E-EN | ECAPA + ResNet (2모델) | 91.78% | 0.9173 | 102,719 / 111,919 | 29.30M | 112.20 MiB | - |
| S-R | ReDimNet2-B2 단독 | 91.75% | 0.9171 | 102,685 / 111,919 | 2.57M | 9.84 MiB | - |
| E-RN | ReDim + ResNet (2모델) | 91.75% | 0.9171 | 102,686 / 111,919 | 26.07M | 99.81 MiB | - |
| S-N | AudioResNet-50 단독 | 91.17% | 0.9112 | 102,032 / 111,919 | 23.50M | 89.97 MiB | NaN 클램프 수정 후 정상치 |
| *W-EER* | *(과거 참고) 가중 앙상블 .45/.40/.15* | 92.54% | 0.9250 | 103,565 / 111,919 | 31.87M | 122.04 MiB | 개발셋 탐색치 (균등 원칙 준수를 위해 E-REN 채택) |

### 2. 91.56% 결함 규명 및 과거 92.48% 정합성
- **결함 원인**: 1차 재현 시 AudioResNet-50의 Mel 변환에서 무음/패딩 1,180건에 대해 `ref_128`이 0이 되어 `div/0 -> NaN` 발생. Numpy 비교 연산에서 `NaN >= 0.50`이 False(0)로 강제 평가되어 1,180건 중 실제 신고자 발화 1,010건이 모두 상황실로 오분류되었음.
- **해소 및 복원**: Librosa와 동일하게 `ref = torch.clamp(ref, min=1e-10)` 하한 클램프를 적용하여 1,180건 NaN을 100% 해소. 1,010건의 정답이 복원되어 정확도가 91.56%(102,474건)에서 **92.46% (103,484건)**로 정상 복원됨.
- 과거 Git 3638dab 보존 기록(92.48%, 약 103,501건 정답)과 순 정답수 17건(0.02%p) 오차 내에서 높은 수준의 정합성을 달성함.

### 3. 실측 전수 정수 혼동 행렬 (3모델 균등 앙상블 92.46%)
- TN (상황실 0 -> 0): 47,729 (88.67%)
- FP (상황실 0 -> 1): 6,101 (11.33%)
- FN (신고자 1 -> 0): 2,334 (4.02%)
- TP (신고자 1 -> 1): 55,755 (95.98%)
- 정답 합계: 103,484건 / 전체 111,919건
